In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os

for dirname in os.listdir('/kaggle/input'):
    print(dirname)
    
base_path = '/kaggle/input/intel-image-classification'
for item in os.listdir(base_path):
    print(item)

notebooks


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/intel-image-classification'

In [2]:
import os

base_path = '/kaggle/input/intel-image-classification'
for item in os.listdir(base_path):
    print(item)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/intel-image-classification'

In [3]:
import os
print(os.listdir('/kaggle/input'))

['datasets', 'notebooks']


In [4]:
import os
print(os.listdir('/kaggle/input/datasets'))


['puneet6060']


In [5]:
import os
print(os.listdir('/kaggle/input/datasets/puneet6060'))

['intel-image-classification']


In [6]:
import os
path = '/kaggle/input/datasets/puneet6060/intel-image-classification'
print(os.listdir(path))

['seg_train', 'seg_pred', 'seg_test']


In [7]:
import os
train_path = '/kaggle/input/datasets/puneet6060/intel-image-classification/seg_train'
print(os.listdir(train_path))

['seg_train']


In [10]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

TRAIN_DIR = '/kaggle/input/datasets/puneet6060/intel-image-classification/seg_train/seg_train'
TEST_DIR = '/kaggle/input/datasets/puneet6060/intel-image-classification/seg_test/seg_test'

transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=transform)

print("Classes found:", train_dataset.classes)
print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_classes = len(train_dataset.classes)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss/len(train_loader):.4f} Train Accuracy: {train_acc:.2f}%")

torch.save(model.state_dict(), "/kaggle/working/scene_model.pth")
print("Model saved to /kaggle/working/scene_model.pth")

Classes found: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Training images: 14034
Test images: 3000
Using device: cuda
Epoch [1/5] Loss: 0.4112 Train Accuracy: 85.86%
Epoch [2/5] Loss: 0.2777 Train Accuracy: 90.31%
Epoch [3/5] Loss: 0.2483 Train Accuracy: 91.36%
Epoch [4/5] Loss: 0.1793 Train Accuracy: 93.75%
Epoch [5/5] Loss: 0.1734 Train Accuracy: 93.97%
Model saved to /kaggle/working/scene_model.pth


In [11]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total
print(f"Test Accuracy: {test_acc:.2f}%")

Test Accuracy: 82.33%


In [12]:
train_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

print("Train images:", len(train_dataset))
print("Test images:", len(test_dataset))

Train images: 14034
Test images: 3000


In [13]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 8
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    # Check test accuracy every epoch too, so we can see the gap shrink in real time
    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
    test_acc = 100 * test_correct / test_total

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss/len(train_loader):.4f} Train Acc: {train_acc:.2f}% Test Acc: {test_acc:.2f}%")

torch.save(model.state_dict(), "/kaggle/working/scene_model_v2.pth")
print("Model saved to /kaggle/working/scene_model_v2.pth")

Epoch [1/8] Loss: 0.4406 Train Acc: 84.77% Test Acc: 85.63%
Epoch [2/8] Loss: 0.3347 Train Acc: 88.45% Test Acc: 87.70%
Epoch [3/8] Loss: 0.2976 Train Acc: 89.53% Test Acc: 90.83%
Epoch [4/8] Loss: 0.2755 Train Acc: 90.39% Test Acc: 85.13%
Epoch [5/8] Loss: 0.2743 Train Acc: 90.51% Test Acc: 87.90%
Epoch [6/8] Loss: 0.2422 Train Acc: 91.41% Test Acc: 89.93%
Epoch [7/8] Loss: 0.2294 Train Acc: 92.10% Test Acc: 91.27%
Epoch [8/8] Loss: 0.2163 Train Acc: 92.28% Test Acc: 90.90%
Model saved to /kaggle/working/scene_model_v2.pth


In [14]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Reduces learning rate by half every 5 epochs
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

best_test_acc = 0.0
num_epochs = 12

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
    test_acc = 100 * test_correct / test_total

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1}/{num_epochs}] LR: {current_lr:.5f} Loss: {running_loss/len(train_loader):.4f} Train Acc: {train_acc:.2f}% Test Acc: {test_acc:.2f}%")

    # Checkpoint: only save if this is the best test accuracy so far
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), "/kaggle/working/scene_model_best.pth")
        print(f"  -> New best model saved (Test Acc: {test_acc:.2f}%)")

    scheduler.step()

print(f"\nBest test accuracy achieved: {best_test_acc:.2f}%")
print("Best model saved to /kaggle/working/scene_model_best.pth")

Epoch [1/12] LR: 0.00100 Loss: 0.4549 Train Acc: 84.09% Test Acc: 75.70%
  -> New best model saved (Test Acc: 75.70%)
Epoch [2/12] LR: 0.00100 Loss: 0.3301 Train Acc: 88.65% Test Acc: 88.17%
  -> New best model saved (Test Acc: 88.17%)
Epoch [3/12] LR: 0.00100 Loss: 0.3113 Train Acc: 89.48% Test Acc: 87.23%
Epoch [4/12] LR: 0.00100 Loss: 0.2693 Train Acc: 90.58% Test Acc: 90.17%
  -> New best model saved (Test Acc: 90.17%)
Epoch [5/12] LR: 0.00100 Loss: 0.2573 Train Acc: 91.27% Test Acc: 88.73%
Epoch [6/12] LR: 0.00050 Loss: 0.1837 Train Acc: 93.57% Test Acc: 91.87%
  -> New best model saved (Test Acc: 91.87%)
Epoch [7/12] LR: 0.00050 Loss: 0.1643 Train Acc: 94.16% Test Acc: 92.27%
  -> New best model saved (Test Acc: 92.27%)
Epoch [9/12] LR: 0.00050 Loss: 0.1494 Train Acc: 94.57% Test Acc: 92.30%
  -> New best model saved (Test Acc: 92.30%)
Epoch [10/12] LR: 0.00050 Loss: 0.1453 Train Acc: 94.78% Test Acc: 91.37%
Epoch [11/12] LR: 0.00025 Loss: 0.1033 Train Acc: 96.38% Test Acc: 92.27